# 🗡️ Hey Aragorn Wake Word Training

Train a custom wake word model using micro-wake-word.

**Run each cell in order. There are two required restarts — the notebook tells you exactly when.**

## Step 0: Free Disk Space

Run this first. Removes ~2–3 GB of unused system files.

In [ ]:
import shutil

for path in ['/usr/share/doc', '/usr/share/man', '/usr/share/locale',
             '/usr/lib/google-cloud-sdk', '/usr/local/android-sdk',
             '/usr/local/julia-1.9.4', '/content/sample_data']:
    shutil.rmtree(path, ignore_errors=True)

!apt-get clean -qq
!apt-get autoremove -y -qq
!pip cache purge -q 2>/dev/null

print("Disk after cleanup:")
!df -h / | tail -1


## Step 1: Check GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")


## Step 2a: Pin numpy & scipy

⚠️ **Only numpy and scipy are installed here.**

A 'Restart required' popup will appear when it finishes — **click it immediately**. Nothing else is in this cell so timing doesn't matter.

After restart → run **Step 2b**.

In [ ]:
!pip uninstall -y numpy scipy 2>/dev/null
!find /usr/local/lib/python3.*/dist-packages -name '*.pyc' -path '*/numpy/*' -delete 2>/dev/null
!find /usr/local/lib/python3.*/dist-packages -name '*.pyc' -path '*/scipy/*' -delete 2>/dev/null
!pip install --force-reinstall --no-cache-dir numpy==1.26.4 scipy==1.13.1
print("\n\u2705 numpy + scipy installed.")
print("\u26a0\ufe0f  Click the Restart button that just appeared above now.")


## Step 2b: Install All Other Packages

Run after the restart from Step 2a.

⚠️ When this cell finishes, **restart once more** (Runtime → Restart session), then continue from Step 3.

In [ ]:
print("Installing espeak-ng...")
!apt-get install -y -q espeak-ng libespeak-ng-dev

# Full output intentional — failures must be visible
print("\nInstalling piper-tts...")
!pip install piper-tts
!python -c "from piper import PiperVoice, SynthesisConfig; print('piper: OK')"

print("\nRemoving conflicting pre-installs...")
# tf-keras intentionally NOT listed — it is the Keras 2 compat layer,
# but microWakeWord requires Keras 3 so we leave TF to manage it
!pip uninstall -y jax jaxlib tensorstore tensorflow-decision-forests tensorflow-text opencv-python opencv-python-headless opencv-contrib-python shap ydf grain pytensor xarray-einstats rasterio tobler cupy-cuda12x 2>/dev/null

print("\nInstalling TensorFlow...")
!pip install --quiet tensorflow==2.16.2 protobuf==4.25.3 ml-dtypes==0.3.2 2>/dev/null

print("\nInstalling remaining deps...")
!pip install --quiet onnxruntime pyyaml datasets mmap-ninja tqdm audiomentations webrtcvad-wheels huggingface_hub 2>/dev/null

print("\n\u2705 All packages installed!")
print("\n\u26a0\ufe0f  Restart the runtime now: Runtime \u2192 Restart session (Ctrl+M .)")
print("   Then continue from Step 3.")


## ⚠️ Restart Required

**Runtime → Restart session** (Ctrl+M .)

Then continue from **Step 3**.

## Step 3: Verify Installs & Clone Repositories

In [ ]:
import subprocess, os, sys, shutil

print(f"Python: {sys.version}")

import numpy as np
assert np.__version__ == '1.26.4', f"Bad numpy: {np.__version__}"
print(f"numpy {np.__version__}: OK")

import scipy
from scipy.signal import resample
from scipy.io import wavfile
print(f"scipy {scipy.__version__}: OK")

import tensorflow as tf
print(f"tensorflow {tf.__version__}: OK")

r = subprocess.run(
    [sys.executable, '-c',
     'from piper import PiperVoice, SynthesisConfig; print("piper: OK")'],
    capture_output=True, text=True)
print(r.stdout.strip() if r.returncode == 0 else f"piper FAILED:\n{r.stderr}")

# Always re-clone fresh — a previous session may have left a patched copy
if os.path.exists('microWakeWord'):
    shutil.rmtree('microWakeWord')
subprocess.run(['git', 'clone',
    'https://github.com/kahrendt/microWakeWord.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
    'microWakeWord'], check=True)

# The ONLY patch: wrap validate_nonstreaming in try/except so a Keras 3
# shape mismatch in model.evaluate() doesn't crash the entire training run.
# The model files (mixednet.py etc.) are NOT touched — tf.keras.ops is
# correct Keras 3 API and must stay as-is.
with open('microWakeWord/microwakeword/train.py', 'a') as f:
    f.write('''

# ── Keras 3 compat: skip validate_nonstreaming on shape errors ────────────────
_orig_validate_nonstreaming = validate_nonstreaming

def validate_nonstreaming(model, config, data_processor):
    try:
        return _orig_validate_nonstreaming(model, config, data_processor)
    except Exception as _e:
        print(f"  [validate_nonstreaming skipped: {type(_e).__name__}]")
        return []
''')
print("microWakeWord patched + OK")

if not os.path.exists('piper-sample-generator'):
    subprocess.run(['git', 'clone',
        'https://github.com/rhasspy/piper-sample-generator.git'], check=True)
print("piper-sample-generator: OK")

print("\n\u2705 All verified and ready!")


## Step 4: Download Piper Voice Model

In [ ]:
import os, urllib.request

os.makedirs('piper-sample-generator/models', exist_ok=True)
url  = ('https://github.com/rhasspy/piper-sample-generator/releases/'
        'download/v2.0.0/en_US-libritts_r-medium.pt')
path = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'

if not os.path.exists(path):
    print('Downloading...')
    urllib.request.urlretrieve(url, path)
    print('\u2705 Done!')
else:
    print('\u2705 Already exists')


## Step 5: Configure

Adjust `TARGET_WORD` until the test sample in Step 6 sounds right.

Tips: underscores between syllables (`hey_air_uh_gorn`), `sh`/`ch`/`th` for those sounds, `ee`/`oo` for long vowels.

In [ ]:
TARGET_WORD    = 'hey_air_uh_gorn'
NUM_SAMPLES    = 1000
TRAINING_STEPS = 10000

print(f'Word:    {TARGET_WORD}')
print(f'Samples: {NUM_SAMPLES}')
print(f'Steps:   {TRAINING_STEPS}')


## Step 6: Generate Test Sample

In [ ]:
import subprocess, os, sys
from IPython.display import Audio, display

os.makedirs('generated_samples', exist_ok=True)
print('Generating 1 test sample...')

env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD, '--max-samples', '1', '--batch-size', '1',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)

if result.returncode == 0:
    wavs = sorted([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    if wavs:
        print(f'Phonetic spelling: "{TARGET_WORD}"')
        print('Adjust TARGET_WORD in Step 5 if this does not sound right.\n')
        display(Audio(os.path.join('generated_samples', wavs[0])))
    else:
        print('No .wav files found.')
else:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)


## Step 7: Generate All Samples

In [ ]:
import subprocess, os, sys

print(f'Generating {NUM_SAMPLES} samples...')
env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD, '--max-samples', str(NUM_SAMPLES), '--batch-size', '100',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)

if result.returncode == 0:
    count = len([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    print(f'\u2705 Generated {count} samples!')
else:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)


## Step 8: Download Augmentation Data

In [ ]:
import os

os.makedirs('mit_rirs', exist_ok=True)
if not os.listdir('mit_rirs'):
    print('Downloading RIRs + background noises (~1.3 GB)...')
    !wget --progress=bar:force -O /tmp/rirs_noises.zip https://www.openslr.org/resources/28/rirs_noises.zip
    !unzip -q /tmp/rirs_noises.zip -d mit_rirs
    print('Extracted!')
else:
    print('Already downloaded')

noise_dir = 'mit_rirs/RIRS_NOISES/pointsource_noises'
if os.path.exists(noise_dir):
    n = len([f for f in os.listdir(noise_dir) if f.endswith('.wav')])
    print(f'\u2705 {n} background noise files ready')
else:
    print('\u274c pointsource_noises not found')


## 🧹 Disk Cleanup (After Step 8)

In [ ]:
import os, subprocess, sys
if os.path.exists('/tmp/rirs_noises.zip'):
    os.remove('/tmp/rirs_noises.zip')
    print('Deleted /tmp/rirs_noises.zip')
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], capture_output=True)
print('pip cache purged')
!df -h / | tail -1


## Step 9: Generate Spectrograms

In [ ]:
import os, sys
if 'microWakeWord' not in sys.path:
    sys.path.insert(0, 'microWakeWord')

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

print('Generating spectrograms...')

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['mit_rirs/RIRS_NOISES/pointsource_noises'],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

# mmap is saved directly here — this is also what the config points to
MMAP_OUT = 'generated_augmented_features/training/wakeword_mmap'
os.makedirs(os.path.dirname(MMAP_OUT), exist_ok=True)

spectrograms = SpectrogramGeneration(
    clips=clips, augmenter=augmenter, slide_frames=10, step_ms=10,
)

RaggedMmap.from_generator(
    out_dir=MMAP_OUT,
    sample_generator=spectrograms.spectrogram_generator(split='train', repeat=2),
    batch_size=100,
    verbose=True,
)
print(f'\u2705 Spectrograms saved to {MMAP_OUT}')


## Step 10: Download Negative Datasets

In [ ]:
import os, zipfile
from huggingface_hub import hf_hub_download, list_repo_files

print('Discovering negative datasets...')
os.makedirs('negative_datasets', exist_ok=True)

repo_id, repo_type = 'kahrendt/microwakeword', 'dataset'
zip_files = [f for f in list_repo_files(repo_id, repo_type=repo_type)
             if f.endswith('.zip')]
print(f'Found: {zip_files}')

for fname in zip_files:
    base    = os.path.splitext(os.path.basename(fname))[0]
    out_dir = f'negative_datasets/{base}'
    if not os.path.exists(out_dir):
        print(f'Downloading {fname}...')
        local = hf_hub_download(repo_id=repo_id, filename=fname,
                                repo_type=repo_type)
        with zipfile.ZipFile(local, 'r') as zf:
            zf.extractall('negative_datasets')
        print(f'  done: {base}')
    else:
        print(f'  exists: {base}')

neg_dirs = [d for d in os.listdir('negative_datasets')
            if os.path.isdir(f'negative_datasets/{d}')]
print(f'\n\u2705 Directories: {neg_dirs}')


## 🧹 Disk Cleanup (After Step 10)

In [ ]:
import os, shutil

hf_cache = os.path.expanduser('~/.cache/huggingface')
if os.path.exists(hf_cache):
    size = sum(os.path.getsize(os.path.join(dp, f))
               for dp, _, fs in os.walk(hf_cache) for f in fs)
    shutil.rmtree(hf_cache)
    print(f'Deleted HuggingFace cache ({size/1e9:.1f} GB freed)')
else:
    print('No HuggingFace cache found')
!df -h / | tail -1


## Step 11: Create Training Config

In [ ]:
import yaml, os

# Filter __MACOSX and other macOS zip metadata dirs
neg_dirs   = sorted([d for d in os.listdir('negative_datasets')
                     if os.path.isdir(f'negative_datasets/{d}')
                     and not d.startswith('__')])
eval_dirs  = [d for d in neg_dirs if 'eval' in d]
train_dirs = [d for d in neg_dirs if 'eval' not in d]
print(f'Train negatives : {train_dirs}')
print(f'Eval  negatives : {eval_dirs}')

neg_features = []
for d in train_dirs:
    neg_features.append({
        'features_dir': f'negative_datasets/{d}',
        'sampling_weight': 10.0, 'penalty_weight': 1.0,
        'truth': False, 'truncation_strategy': 'random', 'type': 'mmap'})
for d in eval_dirs:
    neg_features.append({
        'features_dir': f'negative_datasets/{d}',
        'sampling_weight': 0.0, 'penalty_weight': 1.0,
        'truth': False, 'truncation_strategy': 'split', 'type': 'mmap'})

# Point directly to the mmap directory, not its parent folder.
# Using the parent caused "No spectrograms found" and 0% recall every run.
pos_feature = {
    'features_dir': 'generated_augmented_features/training/wakeword_mmap',
    'sampling_weight': 2.0, 'penalty_weight': 1.0,
    'truth': True, 'truncation_strategy': 'truncate_start', 'type': 'mmap'}

config = {
    'window_step_ms': 10,
    'train_dir': 'trained_models/wakeword',
    'spectrogram_length': 204,
    'stride': 3,
    'features': [pos_feature] + neg_features,
    'training_steps': [TRAINING_STEPS],
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    'learning_rates': [0.001],
    'batch_size': 128,
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.9,
    'minimization_metric': '',
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print(f'\u2705 Config saved. Positive mmap: {pos_feature["features_dir"]}')


## Step 12: Train Model

This takes 1–3 hours on a T4. Go get coffee!

In [ ]:
import subprocess, sys, os, shutil

# Clear any previous partial training run so --restore_checkpoint 0 doesn't error
if os.path.exists('trained_models/wakeword'):
    shutil.rmtree('trained_models/wakeword')
    print('Cleared previous trained_models/wakeword')

print('Starting training...')
print(f'~{TRAINING_STEPS // 10000} hour(s)...')

train_env = {
    **os.environ,
    'TF_FORCE_GPU_ALLOW_GROWTH': 'true',
    'TF_CPP_MIN_LOG_LEVEL': '2',
}

train_cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config=training_parameters.yaml',
    '--train=1',
    '--restore_checkpoint', '0',
    '--test_tf_nonstreaming', '0',
    '--test_tflite_nonstreaming', '0',
    '--test_tflite_nonstreaming_quantized', '0',
    '--test_tflite_streaming', '0',
    '--test_tflite_streaming_quantized', '1',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]

result = subprocess.run(train_cmd, stderr=subprocess.PIPE,
                        text=True, env=train_env)

if result.returncode == 0:
    print('\u2705 Training complete!')
else:
    print('\u274c Training failed. Error output:')
    print(result.stderr[-3000:] if len(result.stderr) > 3000 else result.stderr)


## Step 13: Download Model

In [ ]:
import os
from google.colab import files

model_path = ('trained_models/wakeword/'
              'tflite_stream_state_internal_quant/'
              'stream_state_internal_quant.tflite')

if os.path.exists(model_path):
    print(f'\u2705 Model: {os.path.getsize(model_path)/1024:.1f} KB')
    files.download(model_path)
    print('\n\U0001f389 Done! Check your downloads.')
else:
    print('\u274c Model not found — check training output above')
